# Using Tensorflow Dataset and Modeling


# Imports

In [ ]:
#!pip uninstall tf-keras
# !pip install tensorflow==2.16.1

In [ ]:
import keras
import tensorflow as tf
print("Keras Current Version:", keras.__version__, "Tensorflow Current Version:", tf.__version__)

Keras Current Version: 3.8.0 Tensorflow Current Version: 2.18.0


In [ ]:
import pandas as pd
# burada bir yerde MinMaxScaler'ı import ediniz
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
import tensorflow as tf
scaler=MinMaxScaler()

# Data Preparation

## Task 1: proprocess_data Fonksiyonunda MinMaxScaler'ı Kullanınız.

- Import bölümünde MinMaxScaler'ı import etmeyi unutmayınız.
- preprocess_data fonksiyonu içinde scaler olarak MinMaxScaler'ı kullanınız.
- Sonrasınra train validasyon ayrımını yapınız.

In [ ]:
def preprocess_data(filepath):
    data = pd.read_csv(filepath)

    # Buraya MinMaxScaler ekleyiniz.

    X = scaler.fit_transform(data.drop('Outcome', axis=1))
    y = data['Outcome'].values
    return X, y


X, y = preprocess_data('/content/diabetes.csv')

# train validasyon ayrımını yapınız.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


print("X_train shape:", X_train.shape)

X_train shape: (614, 8)


# Create Tensorflow Dataset


## Task 2: Train Seti ve Validasyon Seti için Tensorflow Dataset Oluşturunuz

In [ ]:
# Train dataset
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))

# Validation dataset
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))

## Task 3: val_dataset'in 1 Gözlemini İnceleyiniz

In [ ]:
for x, y in val_dataset.take(1):
    print("X:", x.numpy())
    print("y:", y.numpy())

X: [0.35294118 0.49246231 0.47540984 0.33333333 0.22458629 0.50670641
 0.15029889 0.36666667]
y: 0


## Task 4: buffer_size'ı tüm veri seti boyutu olarak ve batch'ı 32 olacak şekilde train ve validasyon setlerini biçimlendiriniz. Validasyon seti için sadece batch uyguladığımızı unutmayın.

In [ ]:
train_size = len(X_train)

# Train dataset: shuffle ve batch
train_dataset = train_dataset.shuffle(buffer_size=train_size).batch(32).prefetch(tf.data.AUTOTUNE)

# Validation dataset: sadece batch
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

## Task 5: Validasyon Seti için 1 Gözlemi İnceleyiniz. 10. batch'in feature değerleri nelerdir? Label değeri nedir? Tek tek sayarak inceleyiniz.

In [ ]:
for x,y in train_dataset.take(1):
  print("X:", x.numpy())
  print("y:", y.numpy())
#10.Batch-in feature değerleri ve label değeri 0

X: [[0.05882353 0.62311558 0.60655738 0.36363636 0.         0.414307
  0.00939368 0.15      ]
 [0.05882353 0.46733668 0.45901639 0.11111111 0.         0.33532042
  0.14474808 0.01666667]
 [0.35294118 0.46231156 0.50819672 0.32323232 0.14893617 0.47690015
  0.0029889  0.41666667]
 [0.29411765 0.63316583 0.63934426 0.27272727 0.02600473 0.44113264
  0.15414176 0.31666667]
 [0.         0.47738693 0.6557377  0.45454545 0.10874704 0.54396423
  0.10760034 0.08333333]
 [0.23529412 0.49748744 0.55737705 0.38383838 0.         0.48882265
  0.02860803 0.2       ]
 [0.11764706 0.60301508 0.62295082 0.37373737 0.12411348 0.59165425
  0.05849701 0.13333333]
 [0.35294118 0.48241206 0.         0.         0.         0.35320417
  0.04782237 0.11666667]
 [0.23529412 0.57286432 0.53278689 0.         0.         0.32637854
  0.15115286 0.26666667]
 [0.11764706 0.50251256 0.44262295 0.28282828 0.12411348 0.5633383
  0.1793339  0.05      ]
 [0.29411765 0.93969849 0.62295082 0.27272727 0.24468085 0.64977645
  

# Model

## Task 6: Dersteki modeli 100 epoch sayısı ile tekrar eğitiniz ve loss ve accuracy değerlerini yorumlayınız. Tüm işlemleri tek bir hücrede yapınız.

In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()
history = model.fit(X_train,
                    y_train,
                    batch_size=32,
                    epochs=100,
                    validation_data=(X_val, y_val))

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 833 (3.25 KB)

 Trainable params: 833 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.4863 - loss: 0.6944 - val_accuracy: 0.6429 - val_loss: 0.6777
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6359 - loss: 0.6752 - val_accuracy: 0.6429 - val_loss: 0.6679
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6683 - loss: 0.6590 - val_accuracy: 0.6429 - val_loss: 0.6633
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6284 - loss: 0.6660 - val_accuracy: 0.6429 - val_loss: 0.6588
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6894 - loss: 0.6376 - val_accuracy: 0.6429 - val_loss: 0.6543
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6523 - loss: 0.6510 - val_accuracy: 0.6429 - val_loss: 0.6502
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6879 - loss: 0.6291 - val_accuracy: 0.6429 - val_loss: 0.6460
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6777 - loss: 0.6302 - val_accuracy: 0.6429 - 

In [ ]:
# derste elde edilen değerler
# val_loss: 0.59
# val_accuracy: 0.70

In [ ]:
# Task 6 sonucu:
# Validation Loss:
# Validation Accuracy:
# Yorum:epoch sayı 98 olduğunda daha iyi bir sonuç göre biliriz. Genel olaraq train_accuracy_81, val_accuray_0.77 bence iyi bir model

epoch sayı 100 olduğunda elde etdiğim değerler

Epoch 100/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8021 - loss: 0.4248 - val_accuracy: 0.7338 - val_loss: 0.5191

## Task 7: 5 katmanlı ve nöron sayıları sırasıyla 32, 64, 128, 256, 512 olan bir model kurunuz ve sonuçları değerlendiriniz. Diğer özelliklerde bir değişiklik olmamalı.

In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(64, activation='relu'),
    Dense(128, activation='relu'),
    Dense(256, activation='relu'),
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()
model.compile(loss='binary_crossentropy',
              metrics=['accuracy'])
history = model.fit(X_train,
                    y_train,
                    batch_size=32,
                    epochs=100,
                    validation_data=(X_val, y_val))

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_36 (Dense)                │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 175,841 (686.88 KB)

 Trainable params: 175,841 (686.88 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 92ms/step - accuracy: 0.6436 - loss: 0.6632 - val_accuracy: 0.6623 - val_loss: 0.6633
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6711 - loss: 0.6293 - val_accuracy: 0.7013 - val_loss: 0.6203
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6801 - loss: 0.5879 - val_accuracy: 0.6558 - val_loss: 0.5873
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7005 - loss: 0.5471 - val_accuracy: 0.7078 - val_loss: 0.5596
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7289 - loss: 0.5495 - val_accuracy: 0.7338 - val_loss: 0.5500
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7555 - loss: 0.5057 - val_accuracy: 0.6753 - val_loss: 0.6122
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7495 - loss: 0.5344 - val_accuracy: 0.6753 - val_loss: 0.5754
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7505 - loss: 0.5052 - val_accuracy: 0.5649 - 

In [ ]:
# Yorum:
# Neden?:

**train_accuracy artdı ama aynı zamanda val_loss artdış Bu istemediğimiz bir şey overfitting oldu ve model ezberlemiş olduğu için train_accuracy artıyor sadece**


## Task 8: Acaba Epoch Sayısını Arttırsak bir Faydası Olur mu? 10000 ile deneyelim. Sonuçları yorumlayınız.

In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(64, activation='relu'),
    Dense(128, activation='relu'),
    Dense(256, activation='relu'),
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()
model.compile(loss='binary_crossentropy',
              metrics=['accuracy'])
history = model.fit(X_train,
                    y_train,
                    batch_size=32,
                    epochs=1000,
                    validation_data=(X_val, y_val))

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_42 (Dense)                │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 175,841 (686.88 KB)

 Trainable params: 175,841 (686.88 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.6006 - loss: 0.6792 - val_accuracy: 0.6429 - val_loss: 0.6768
Epoch 2/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6662 - loss: 0.6373 - val_accuracy: 0.6429 - val_loss: 0.7287
Epoch 3/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6415 - loss: 0.6526 - val_accuracy: 0.6883 - val_loss: 0.5975
Epoch 4/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6926 - loss: 0.5722 - val_accuracy: 0.6429 - val_loss: 0.6647
Epoch 5/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7075 - loss: 0.5536 - val_accuracy: 0.6429 - val_loss: 0.6302
Epoch 6/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7350 - loss: 0.5201 - val_accuracy: 0.6883 - val_loss: 0.5959
Epoch 7/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7091 - loss: 0.5254 - val_accuracy: 0.6688 - val_loss: 0.6053
Epoch 8/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7559 - loss: 0.5014 - val_accuracy: 0

In [ ]:
# Yorum:Bence model performansı iyi olmaz çünki model datayı dahada ezberler varyans artar val_loss artar accuray ise şiddetli artar.(#kendi fikrim)
#denedikden sonrada bunu görüyoruz